In [0]:
# dbutils.fs.ls("abfss://ingestion@musedatapipeline.dfs.core.windows.net/ingestion/sample/2026/06/")

In [0]:
from pyspark.sql import SparkSession
import logging as l
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
l.basicConfig(level=l.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = l.getLogger(__name__)

start_time = datetime.now()
logger.info("Bronze layer started")

In [0]:
dbutils.widgets.text("ingestion_date", "")
ingestion_date = dbutils.widgets.get("ingestion_date")

if not ingestion_date:
    from datetime import date
    ingestion_date = date.today().strftime("%Y/%m/%d")

In [0]:

try: 
    # raw_path = "abfss://ingestion@musedatapipeline.dfs.core.windows.net/ingestion/sample//2026/06/14/*.json"
    # raw_path = "abfss://ingestion@musedatapipeline.dfs.core.windows.net/ingestion/sample//2026/06/14/*.json"
    raw_path = f"abfss://ingestion@musedatapipeline.dfs.core.windows.net/muse/{ingestion_date}/*.json"
    df_raw = spark.read \
        .option("multiline", "true") \
        .json(raw_path)
        
    print(f"Raw json file count: {df_raw.count()} ")
    logger.info(f"Raw files imported")
    df_raw = df_raw.withColumn("_ingestion_date", F.current_date())
  
    logger.info(f"Raw files imported on {datetime.now()}")
except Exception as e:
    logger.error(f"Error: {e}")
    raise 


In [0]:
# df_raw = df_raw.withColumn("_ingestion_date", F.current_date())

In [0]:
df_raw.show()

In [0]:

try:
    #explode list into individual lists
    results_df = df_raw.select(F.explode("results").alias("job"), "_ingestion_date")
    logger.info("Results Exploded.")
    #use select to get data into individual columns
    bronze_df = results_df.select("job.*", "_ingestion_date")
    logger.info("Bronze table created.")
    bronze_df.show(5)
except Exception as e:
    logger.error(f"Error: {e}")
    raise


In [0]:
# bronze_df = results_df.select("job.*", "_ingestion_date")

In [0]:
# bronze_df.show()

In [0]:
bronze_df.printSchema()

In [0]:
# spark.sql("DROP TABLE IF EXISTS muse.bronze.job_muse_bronze")


In [0]:
spark.sql("SHOW TABLES IN muse.bronze").show()

In [0]:
try: 
    bronze_df.write \
        .format("delta") \
        .mode("append") \
        .option("path", "abfss://bronze@musedatapipeline.dfs.core.windows.net/bronze/jobs") \
        .saveAsTable("muse.bronze.job_muse_bronze_landing")
except Exception as e:
    logger.error(f"Error: {e}")
    raise
finally:
    logger.info(f"Bronze landing table created.")

In [0]:
# dbutils.fs.ls("abfss://ingestion@musedatapipeline.dfs.core.windows.net/")

In [0]:
# spark.sql("DESCRIBE DETAIL muse.bronze.job_muse_bronze_landing").select("location").show(truncate=False)

In [0]:
%sql
-- drop table if exists muse.bronze.table_job

In [0]:
%sql

          select * from 
          muse.bronze.job_muse_bronze
          
     

In [0]:
logger.info("Bronze layer ingestion completed!!")